In [39]:
import pandas as pd
import numpy as np
import tensorflow as tf
import re
import os
from sklearn.model_selection import train_test_split
from tokenizers import BertWordPieceTokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [40]:
data = pd.read_csv("ara_.txt", sep="\t", header=None, names=["en", "ar"])

def clean(text):
    text = str(text).lower()
    tashkeel_pattern = re.compile(r"[\u064B-\u0652]")
    text = re.sub(tashkeel_pattern, "", text)
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"[^\w\s]", "", text)
    return text.strip()

data["ar_clean"] = data["ar"].apply(clean)
data["en_clean"] = data["en"].apply(clean)


with open("en_raw.txt", "w", encoding="utf-8") as f: f.write("\n".join(data["en_clean"]))
with open("ar_raw.txt", "w", encoding="utf-8") as f: f.write("\n".join(data["ar_clean"]))


In [41]:
en_tokenizer = BertWordPieceTokenizer(lowercase=True)
en_tokenizer.train(files=["en_raw.txt"], vocab_size=5000)
ar_tokenizer = BertWordPieceTokenizer(lowercase=True)
ar_tokenizer.train(files=["ar_raw.txt"], vocab_size=8000)

START_TOKEN = "[#START#]"
END_TOKEN = "[#END#]"
ar_tokenizer.add_special_tokens([START_TOKEN, END_TOKEN])


2

In [42]:
def encode(lang_tokenizer, text_list, add_special=False):
    ids = []
    for text in text_list:
        if add_special: text = f"{START_TOKEN} {text} {END_TOKEN}"
        ids.append(lang_tokenizer.encode(text).ids)
    return ids

max_len = 25
en_seq = pad_sequences(encode(en_tokenizer, data["en_clean"]), maxlen=max_len, padding="post")
ar_seq = pad_sequences(encode(ar_tokenizer, data["ar_clean"], True), maxlen=max_len, padding="post")

en_train, en_val, ar_train, ar_val = train_test_split(en_seq, ar_seq, test_size=0.1, random_state=42)
dec_in_train, dec_out_train = ar_train[:, :-1], ar_train[:, 1:]
dec_in_val, dec_out_val = ar_val[:, :-1], ar_val[:, 1:]

In [43]:
def positional_encoding(position, d_model):
    angle_rads = np.arange(position)[:, np.newaxis] / np.power(10000, (2 * (np.arange(d_model)[np.newaxis, :] // 2)) / np.float32(d_model))
    pos_encoding = np.zeros(angle_rads.shape)
    pos_encoding[:, 0::2], pos_encoding[:, 1::2] = np.sin(angle_rads[:, 0::2]), np.cos(angle_rads[:, 1::2])
    return tf.cast(pos_encoding[np.newaxis, ...], dtype=tf.float32)

In [44]:
class EncoderLayer(Layer):
    def __init__(self, d_model):
        super().__init__()
        self.mha = MultiHeadAttention(num_heads=4, key_dim=d_model)
        self.ffn = tf.keras.Sequential([Dense(512, activation="relu"), Dense(d_model)])
        self.norm1, self.norm2 = LayerNormalization(), LayerNormalization()
        self.dropout1, self.dropout2 = Dropout(0.1), Dropout(0.1)

    def call(self, x, training=False, mask=None):
        attn_mask = mask[:, tf.newaxis, :] if mask is not None else None
        attn = self.mha(x, x, attention_mask=attn_mask)
        x = self.norm1(x + self.dropout1(attn, training=training))
        x = self.norm2(x + self.dropout2(self.ffn(x), training=training))
        return x

In [45]:
class DecoderLayer(Layer):
    def __init__(self, d_model):
        super().__init__()
        self.mha1 = MultiHeadAttention(num_heads=4, key_dim=d_model)
        self.mha2 = MultiHeadAttention(num_heads=4, key_dim=d_model)
        self.ffn = tf.keras.Sequential([Dense(512, activation="relu"), Dense(d_model)])
        self.norm1, self.norm2, self.norm3 = LayerNormalization(), LayerNormalization(), LayerNormalization()
        self.dropout1, self.dropout2, self.dropout3 = Dropout(0.1), Dropout(0.1), Dropout(0.1)

    def call(self, x, enc, training=False, mask=None):
        attn1 = self.mha1(x, x, use_causal_mask=True, attention_mask=mask[:, tf.newaxis, :] if mask is not None else None)
        x = self.norm1(x + self.dropout1(attn1, training=training))
        attn2 = self.mha2(x, enc)
        x = self.norm2(x + self.dropout2(attn2, training=training))
        x = self.norm3(x + self.dropout3(self.ffn(x), training=training))
        return x

In [46]:
d_model = 256
encoder_inputs, decoder_inputs = Input(shape=(max_len,)), Input(shape=(max_len-1,))
enc_emb = Embedding(en_tokenizer.get_vocab_size(), d_model, mask_zero=True)(encoder_inputs) + positional_encoding(max_len, d_model)
dec_emb = Embedding(ar_tokenizer.get_vocab_size(), d_model, mask_zero=True)(decoder_inputs) + positional_encoding(max_len-1, d_model)

x = Dropout(0.1)(enc_emb)
for _ in range(2): x = EncoderLayer(d_model)(x)
enc_output = x

x = Dropout(0.1)(dec_emb)
for _ in range(2): x = DecoderLayer(d_model)(x, enc_output)
outputs = Dense(ar_tokenizer.get_vocab_size(), activation="softmax")(x)

model = Model([encoder_inputs, decoder_inputs], outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(0.0004), loss="sparse_categorical_crossentropy", metrics=["accuracy"])


In [47]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

model.fit(
    [en_train, dec_in_train], dec_out_train,
    validation_data=([en_val, dec_in_val], dec_out_val),
    epochs=100, # Increased
    batch_size=128,
    callbacks=[early_stop, lr_scheduler]
)

Epoch 1/100
76/76 ━━━━━━━━━━━━━━━━━━━━ 59s 410ms/step - accuracy: 0.7474 - loss: 3.0298 - val_accuracy: 0.7904 - val_loss: 1.7588 - learning_rate: 4.0000e-04
Epoch 2/100
76/76 ━━━━━━━━━━━━━━━━━━━━ 11s 147ms/step - accuracy: 0.7939 - loss: 1.6786 - val_accuracy: 0.7941 - val_loss: 1.6540 - learning_rate: 4.0000e-04
Epoch 3/100
76/76 ━━━━━━━━━━━━━━━━━━━━ 12s 154ms/step - accuracy: 0.7965 - loss: 1.6067 - val_accuracy: 0.7956 - val_loss: 1.6215 - learning_rate: 4.0000e-04
Epoch 4/100
76/76 ━━━━━━━━━━━━━━━━━━━━ 11s 146ms/step - accuracy: 0.7972 - loss: 1.5644 - val_accuracy: 0.7960 - val_loss: 1.5932 - learning_rate: 4.0000e-04
Epoch 5/100
76/76 ━━━━━━━━━━━━━━━━━━━━ 11s 145ms/step - accuracy: 0.7978 - loss: 1.5292 - val_accuracy: 0.7966 - val_loss: 1.5711 - learning_rate: 4.0000e-04
Epoch 6/100
76/76 ━━━━━━━━━━━━━━━━━━━━ 11s 139ms/step - accuracy: 0.7989 - loss: 1.4918 - val_accuracy: 0.7980 - val_loss: 1.5539 - learning_rate: 4.0000e-04
Epoch 7/100
76/76 ━━━━━━━━━━━━━━━━━━━━ 10s 137ms/ste

In [48]:
# Fine Tuning to fix the specific pronoun mistakes.
balanced_fixes = pd.DataFrame([
    {"en_clean": "i love him", "ar_clean": "احبه"},
    {"en_clean": "i love her", "ar_clean": "احبها"},
    {"en_clean": "i love them", "ar_clean": "احبهم"},
    {"en_clean": "i love it", "ar_clean": "احب ذلك"},
    {"en_clean": "he loves her", "ar_clean": "هو يحبها"},
    {"en_clean": "she loves him", "ar_clean": "هي تحبه"}
])

repair_data = pd.concat([balanced_fixes] * 300, ignore_index=True)

anchor_data = data.sample(1000)
final_repair_df = pd.concat([repair_data, anchor_data], ignore_index=True)

en_seq_rep = pad_sequences(encode(en_tokenizer, final_repair_df["en_clean"]), maxlen=max_len, padding="post")
ar_seq_rep = pad_sequences(encode(ar_tokenizer, final_repair_df["ar_clean"], True), maxlen=max_len, padding="post")
dec_in_rep, dec_out_rep = ar_seq_rep[:, :-1], ar_seq_rep[:, 1:]

model.compile(optimizer=tf.keras.optimizers.Adam(5e-6), loss="sparse_categorical_crossentropy")

print("Starting Fine-Tuning...")
model.fit([en_seq_rep, dec_in_rep], dec_out_rep, epochs=8, batch_size=32)

Starting Fine-Tuning...
Epoch 1/8
88/88 ━━━━━━━━━━━━━━━━━━━━ 48s 219ms/step - accuracy: 0.9315 - loss: 0.3487
Epoch 2/8
88/88 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - accuracy: 0.9377 - loss: 0.2947
Epoch 3/8
88/88 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - accuracy: 0.9427 - loss: 0.2724
Epoch 4/8
88/88 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - accuracy: 0.9458 - loss: 0.2600
Epoch 5/8
88/88 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - accuracy: 0.9483 - loss: 0.2530
Epoch 6/8
88/88 ━━━━━━━━━━━━━━━━━━━━ 4s 42ms/step - accuracy: 0.9507 - loss: 0.2462
Epoch 7/8
88/88 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - accuracy: 0.9537 - loss: 0.2396
Epoch 8/8
88/88 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - accuracy: 0.9554 - loss: 0.2349


In [49]:
def beam_translate(sentence, beam_width=3):
    sentence = clean(sentence)
    enc_input = pad_sequences([en_tokenizer.encode(sentence).ids], maxlen=max_len, padding='post')
    start_id, end_id = ar_tokenizer.token_to_id(START_TOKEN), ar_tokenizer.token_to_id(END_TOKEN)
    candidates = [([start_id], 0.0)]

    for _ in range(max_len - 1):
        all_candidates = []
        for seq, score in candidates:
            if seq[-1] == end_id:
                all_candidates.append((seq, score)); continue
            preds = model([enc_input, pad_sequences([seq], maxlen=max_len-1, padding='post')], training=False)
            probs = preds[0, len(seq)-1, :].numpy()
            top_k = np.argsort(probs)[-beam_width:]
            for idx in top_k: all_candidates.append((seq + [idx], score + np.log(probs[idx] + 1e-10)))
        candidates = sorted(all_candidates, key=lambda x: x[1], reverse=True)[:beam_width]
        if all(c[0][-1] == end_id for c in candidates): break
    return ar_tokenizer.decode([i for i in candidates[0][0] if i not in [start_id, end_id]])


In [50]:
text = "Don't worry"
print(f"Input: {text} -> Output: {beam_translate(text)}")

Input: Don't worry -> Output: لا تقلق


In [51]:
print(f"Input: how are you? -> Output: {beam_translate('how are you?')}")

Input: how are you? -> Output: كيف حالك


In [52]:
print("\n--- Testing the Fixes ---")
test_queries = ["I love him", "I love her", "I love them", "I love it"]
for q in test_queries:
    print(f"Input: {q} -> Output: {beam_translate(q)}")


--- Testing the Fixes ---
Input: I love him -> Output: احبه
Input: I love her -> Output: احبها
Input: I love them -> Output: احبهم
Input: I love it -> Output: احب ذلك


In [53]:
# Save these successful weights immediately
model.save_weights("translator_final.weights.h5")
print("Weights saved successfully!")

Weights saved successfully!
